In [1]:
import gymnasium as gym
import numpy as np

## Value iterator

In [2]:
def value_iteration(entorno, gamma_p, theta_p):
    
    """
    Implementa el algoritmo Value Iteration (Iteración de Valor).

    Fundamento teórico:
    -------------------
    Aplica la ecuación de Bellman iterativamente sobre todos los estados
    hasta convergencia. En cada barrida actualiza:

        V(s) = max_a { Σ P(s'|s,a) · [R + γ · V(s')] }

    Al converger, extrae la política óptima:
        π*(s) = argmax_a Q(s,a)

    Parámetros:
    -----------
    entorno : gym.Env  — entorno Gym con acceso a entorno.P (modelo del entorno)
    gamma   : float    — factor de descuento (0 < γ ≤ 1)
    theta   : float    — criterio de convergencia (umbral de cambio mínimo)

    Retorna:
    --------
    V       : np.ndarray — función de valor óptima para cada estado
    politica: np.ndarray — política óptima (acción por estado)
    iteraciones: int     — cantidad de iteraciones hasta convergencia
    """

    iteracion = 0
    modelo = entorno.unwrapped.P
    num_estados = entorno.observation_space.n
    num_acciones = entorno.action_space.n
    V = np.zeros(num_estados)
    
    while True:
        delta = 0.0

        for s in range(num_estados):
            valor_anterior = V[s]
            valores_q = np.zeros(num_acciones)

            for a in range(num_acciones):
                for prob, siguiente, recompensa, terminado in modelo[s][a]:
                    valores_q[a] += prob * (recompensa + gamma_p * V[siguiente])

            # Guardamos el máximo valor encontrado entre las acciones
            V[s] = max(valores_q)

            # Calcular el cambio máximo para la condición de parada
            delta = max(delta, abs(valor_anterior - V[s]))

        iteracion += 1
        ##print(f"Iteración {iteracion}: V = {np.round(V, 2)}")

        if delta < theta_p:
            print("¡El algoritmo ha convergido!")
            break

    politica_estados = {}
    for s in range(num_estados):
        valores_q = np.zeros(num_acciones)
        for a in range(num_acciones):
            for prob, siguiente, recompensa, terminado in modelo[s][a]:
                valores_q[a] += prob * (recompensa + gamma_p * V[siguiente])

        politica_estados[s] = np.argmax(valores_q)
    return V, politica_estados



def impresion_matriz_v_y_politica(nombre, matriz, policy):
    print(f"Matriz {nombre} de Valores:")
    print(np.round(matriz.reshape(4, 4), 4))
    print("\nPolítica Óptima:")
    for i in policy:
        print(i, "= ", policy.get(i))


In [3]:
# Ejecución de Frozen Lake en un entorno Determinista.

In [7]:
env = gym.make("FrozenLake-v1", render_mode="human", is_slippery=False)
observation, info = env.reset()

episode_over = False
total_reward = 0
gamma = 0.99
theta = 1e-8

matriz_costos_v, json_politica = value_iteration(env, gamma, theta)
tipo_matriz = "Determinista"
impresion_matriz_v_y_politica(tipo_matriz, matriz_costos_v, json_politica)


estado_actual = observation
while not episode_over:
    # 0: Move left 1: Move down 2: Move right 3: Move top
    action = json_politica.get(estado_actual)
    estado_actual, reward, terminated, truncated, info = env.step(action)

    total_reward += reward

    episode_over = terminated or truncated

print(f"\nEjecución finalizada! Recompensa total: {total_reward}")
env.close()

¡El algoritmo ha convergido!
Matriz Determinista de Valores:
[[0.951  0.9606 0.9703 0.9606]
 [0.9606 0.     0.9801 0.    ]
 [0.9703 0.9801 0.99   0.    ]
 [0.     0.99   1.     0.    ]]

Política Óptima:
0 =  1
1 =  2
2 =  1
3 =  0
4 =  1
5 =  0
6 =  1
7 =  0
8 =  2
9 =  1
10 =  1
11 =  0
12 =  0
13 =  2
14 =  2
15 =  0

Ejecución finalizada! Recompensa total: 1


In [8]:
conclusiones

NameError: name 'conclusiones' is not defined

In [ ]:
# Ejecución de Frozen Lake en un entorno Estocástico.

In [9]:
env = gym.make("FrozenLake-v1", render_mode="human", is_slippery=True)
observation, info = env.reset()

episode_over = False
total_reward = 0
gamma = 0.99
theta = 1e-8

matriz_costos_v, json_politica = value_iteration(env, gamma, theta)
tipo_matriz = "Estocástico"
impresion_matriz_v_y_politica(tipo_matriz, matriz_costos_v, json_politica)


estado_actual = observation
while not episode_over:
    # 0: Move left 1: Move down 2: Move right 3: Move top

    action = json_politica.get(estado_actual)
    estado_actual, reward, terminated, truncated, info = env.step(action)

    total_reward += reward

    episode_over = terminated or truncated

print(f"\nEjecución finalizada! Recompensa total: {total_reward}")
env.close()

¡El algoritmo ha convergido!
Matriz Estocástico de Valores:
[[0.542  0.4988 0.4707 0.4569]
 [0.5585 0.     0.3583 0.    ]
 [0.5918 0.6431 0.6152 0.    ]
 [0.     0.7417 0.8628 0.    ]]

Política Óptima:
0 =  0
1 =  3
2 =  3
3 =  3
4 =  0
5 =  0
6 =  0
7 =  0
8 =  3
9 =  1
10 =  0
11 =  0
12 =  0
13 =  2
14 =  1
15 =  0

Ejecución finalizada! Recompensa total: 0


In [ ]:
conclusiones

# Q-LEARNING

In [ ]:
def aprendizaje_q(
    entorno,
    episodios=10000,
    alfa=0.1,
    gamma=0.99,
    epsilon=1.0,
    decaimiento_epsilon=0.999,):

    
    num_estados = entorno.observation_space.n
    num_acciones = entorno.action_space.n

    Q = np.zeros((num_estados, num_acciones))
    recompensas_por_episodio = []

    for episodio in range(episodios):

        info_estado = entorno.reset()
        estado = info_estado[0]

        recompensa_total = 0
        terminado = False

        while not terminado:

            if np.random.uniform(0, 1) < epsilon:
                accion = entorno.action_space.sample() 
            else:
                accion = np.argmax(Q[estado, :])  

            resultado_accion = entorno.step(accion)

            siguiente_estado, recompensa, finalizado, truncado, _ = resultado_accion
            terminado = finalizado or truncado

            # 3. Actualizar Q mediante la ecuación de aprendizaje (Diferencia Temporal)
            # Calculamos el objetivo DT (Diferencia Temporal)
            # mejor_siguiente_accion = np.argmax(Q[siguiente_estado, :])
            # objetivo_dt = recompensa + gamma * Q[siguiente_estado, mejor_siguiente_accion] * (not terminado)
            # error_dt = objetivo_dt - Q[estado, accion]
            # Q[estado, accion] += alfa * error_dt
            # ESTA ERA LA FORMA ANTERIOR DE CALCULAR, ES LO MISMO LA DE ABAJO PERO EN UNA LINEA

            Q[estado, accion] += alfa * (recompensa + gamma * np.max(Q[siguiente_estado, :]) * (not terminado) - Q[estado, accion])

            estado = siguiente_estado
            recompensa_total += recompensa

        
        epsilon = max(0.01, epsilon * decaimiento_epsilon)
        recompensas_por_episodio.append(recompensa_total)


    politica = np.argmax(Q, axis=1)

    return Q, politica, recompensas_por_episodio


# DOS COSAS A TENER EN CUENTA
# la ecuacion q puede estar implementada en una linea o en varias, decidir eso
# el decaimiento de epsilon se puede hacer al comienzo como en el pseudocodigo pero conlleva que
# o se evite en el primer episodio o que epsilon no sea 100 en el primer intento, lo cual no se si es correcto

# CART POLE

Tiene dos variantes v0 y v1

el agente tiene que mantenerse con el palo en equilibrio lo mas que pueda
 - en v0 -> 200 pasos y umbral de exito 200
 - en v1 -> 500 pasos y umbal de exito 500

dos formas de recompensa
 1. El agente gana puntos por cada paso que sobrevive
 2. El agente no gana puntos pero pierde si se muere

dos formas de que en teoria tienen el mismo resultado pero estaria interesante comparar ambos

In [ ]:

env = gym.make("CartPole-v1", render_mode="human")
env
# <TimeLimit<OrderEnforcing<PassiveEnvChecker<CartPoleEnv<CartPole-v1>>>>>
env.reset(seed=123, options={"low": -0.1, "high": 0.1})  # default low=-0.05, high=0.05
# (array([ 0.03647037, -0.0892358 , -0.05592803, -0.06312564], dtype=float32), {})

observation, info = env.reset()
# observation: what the agent can "see" 
# info: extra debugging information (usually not needed for basic learning)

print(f"Starting observation: {observation}")

Starting observation: [-0.03240941  0.03120945  0.0423345  -0.02234256]


: 